In [1]:
# install (CPU) — run once
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install --quiet ftfy regex tqdm
!pip install --quiet git+https://github.com/openai/CLIP.git
!pip install --quiet fastapi uvicorn python-multipart pandas nest-asyncio pyngrok pillow requests scikit-learn

# upload dataset
from google.colab import files
uploaded = files.upload()  # choose landmark_embeddings2.csv


  Preparing metadata (setup.py) ... done


Saving landmark_embeddings2.csv to landmark_embeddings2 (3).csv


In [2]:
import pandas as pd
import ast
df = pd.read_csv("landmark_embeddings2.csv")
# convert embedding string to list if needed
def parse_embeddings(x):
    if isinstance(x, str):
        try:
            return list(map(float, ast.literal_eval(x)))
        except Exception as e:
            # handle comma-separated without brackets
            return list(map(float, x.strip().split()))
    elif isinstance(x, list):
        return x
    else:
        return []

df['embeddings'] = df['embeddings'].apply(parse_embeddings)
print("Loaded", len(df), "landmarks")
# quick check
print(df.head(2).to_dict(orient='records'))


Loaded 247 landmarks
[{'name': 'walchand institue of technology', 'embeddings': [-0.0067755854688584805, 0.01574934832751751, -0.015294444747269154, 0.003602272132411599, 0.0025896423030644655, -0.001109969336539507, -0.029912397265434265, 0.007581715006381273, -0.05093279480934143, -0.006211592350155115, -0.0021594827994704247, 0.0295614842325449, -0.009858863428235054, -0.02942941151559353, 0.0024550026282668114, 0.027264753356575966, 0.1001855805516243, 0.011543883010745049, -0.025227321311831474, -0.041860826313495636, 0.05164273828268051, 0.0013603209517896175, 0.027469061315059662, -0.0211963951587677, 0.02733219787478447, 0.06233630329370499, 0.06404034048318863, 0.00776028074324131, -0.032295431941747665, -0.023059656843543053, -0.005299779586493969, -0.02133123204112053, 0.01641455478966236, 0.00953561719506979, -0.021432306617498398, -0.006257574539631605, -0.01979251392185688, 0.011970062740147114, 0.007065776269882917, -0.06298806518316269, -0.005394866224378347, 0.00533528

In [3]:
import clip
import torch
import numpy as np
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from io import BytesIO
from pyngrok import ngrok
import uvicorn
import nest_asyncio


#  Setup

device = "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# Ngrok (replace with your token)
ngrok.set_auth_token("3BmgGjvHVi1wAfwLlR65tOW7zqY_5Ymos6PwXDjcBtfVhYQfc")

# FastAPI init
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

#  Helper functions
def image_to_embedding(pil_image):
    """Converts PIL image to normalized CLIP embedding"""
    try:
        img = preprocess(pil_image).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = model.encode_image(img).cpu().numpy()[0]
        emb = emb / np.linalg.norm(emb)
        return emb
    except Exception as e:
        print(f" Embedding error: {e}")
        return None

# Preload known landmarks
landmarks = []
for _, row in df.iterrows():  # assumes df already loaded
    emb = np.array(row['embeddings'], dtype=float)
    if np.linalg.norm(emb) != 0:
        emb = emb / np.linalg.norm(emb)
    landmarks.append({
        "name": row.get("name"),
        "wikiLink": row.get("wikipedialink", ""),
        "embedding": emb
    })

# Image Search Endpoint
@app.post("/search")
async def search(file: UploadFile = File(...)):
    try:
        data = await file.read()
        pil_img = Image.open(BytesIO(data)).convert("RGB")
        query_emb = image_to_embedding(pil_img)
        if query_emb is None:
            return {"error": "embedding_failed"}

        best_score = -1.0
        best_hit = None

        for lm in landmarks:
            emb = lm.get("embedding")
            if emb is None or len(emb) == 0:
                continue
            score = float(np.dot(query_emb, emb))
            if score > best_score:
                best_score = score
                best_hit = lm

        # Only accept if similarity >= 0.75
        print(f"Image match similarity: {best_score:.2f}")

        if best_hit and best_score >= 0.75:
            print(f"Matched Landmark: {best_hit['name']} ({best_score:.2f})")
            return {
                "landmarkName": best_hit["name"],
                "wikiLink": best_hit.get("wikiLink", ""),
                "score": round(best_score, 3)
            }

        print("No good match found — below threshold (0.75)")
        return {
            "landmarkName": "Unknown",
            "wikiLink": "",
            "score": round(best_score, 3)
        }

    except Exception as e:
        print(f"Error in search: {e}")
        return {"error": str(e)}

# --- CLIP SERVICE TEXT-TO-SPEECH ENHANCEMENT ---
# Instructions:
# 1. Install dependencies: pip install fastapi uvicorn edge-tts pydantic
# 2. Add the following code blocks to your existing FastAPI app OR run this as a standalone service.

import io
import edge_tts
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# The 'app = FastAPI()' line is already present at the top of the cell.
# Removing the duplicate initialization to avoid conflicts.
# app = FastAPI()

# Enable CORS for local testing if needed
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    voice: str = "en-IN-NeerjaNeural" # Premium natural Indian-English voice

@app.get("/")
async def root():
    return {"status": "AI Service Running", "features": ["Landmark Recognition", "Text-to-Speech"]}

@app.post("/tts")
async def text_to_speech(request: TTSRequest):
    """
    Generates high-quality MP3 audio from text using Microsoft Edge's free TTS service.
    """
    if not request.text:
        raise HTTPException(status_code=400, detail="Text is required")

    print(f"[TTS] Generating speech for: {request.text[:50]}...")

    try:
        communicate = edge_tts.Communicate(request.text, request.voice)
        audio_stream = io.BytesIO()

        # Stream chunks to prevent memory overhead
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                audio_stream.write(chunk["data"])

        audio_stream.seek(0)
        return StreamingResponse(audio_stream, media_type="audio/mpeg")
    except Exception as e:
        print(f"[TTS Error] {str(e)}")
        raise HTTPException(status_code=500, detail=f"TTS Generation Failed: {str(e)}")

# Combine this with your existing /predict or /search routes

# The uvicorn.run() call causes a RuntimeError in Colab due to an already running event loop.
# This block is removed as the server is started correctly using nest_asyncio below.
# if __name__ == "__main__":
#     uvicorn.run(app, host="0.0.0.0", port=8000)

# Start Server
nest_asyncio.apply()
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print("FastAPI server running on port 8000")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)

async def start_server():
    await server.serve()

import asyncio
asyncio.get_event_loop().create_task(start_server())

Public URL: NgrokTunnel: "https://enriqueta-microbeless-lottie.ngrok-free.dev" -> "http://localhost:8000"
FastAPI server running on port 8000


<Task pending name='Task-1' coro=<start_server() running at /tmp/ipykernel_33371/2531061081.py:173>>